In [ ]:
# 清理 storage
from pathlib import Path
from tqdm import tqdm

storage_dir = Path("../storage")
video_storage_dir = storage_dir / "videos"
image_storage_dir = storage_dir / "images"
audio_storage_dir = storage_dir / "audios"
effects_storage_dir = storage_dir / "effects"

for path in tqdm(audio_storage_dir.glob("*"), desc="clean audio storage"):
    path.unlink()

for path in tqdm(image_storage_dir.glob("*"), desc="clean image storage"):
    path.unlink()

for path in tqdm(video_storage_dir.glob("*"), desc="clean video storage"):
    path.unlink()

for path in tqdm(effects_storage_dir.glob("*"), desc="clean effects storage"):
    path.unlink()


In [ ]:
# 初始化 Effect 表：从 components_description.json 加载特效到数据库
import json
import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from sqlalchemy import text
from sqlmodel import Session, SQLModel
from app.database import engine
from app.models.effect import Effect

EFFECTS_JSON = Path("../components_description.json")
DOCS_DIR = Path("../storage/effects")


def pascal_to_kebab(name: str) -> str:
    return re.sub(r'(?<!^)(?=[A-Z])', '-', name).lower()


with open(EFFECTS_JSON, encoding="utf-8") as f:
    effects_data = json.load(f)

SQLModel.metadata.create_all(engine)

with Session(engine) as session:
    session.exec(text("DELETE FROM effect"))
    session.commit()

    for item in effects_data:
        kebab_name = pascal_to_kebab(item["name"])
        doc_path = DOCS_DIR / f"{kebab_name}.md"

        effect = Effect(
            name=item["name"],
            category=item["category"],
            description=item["description"],
            library="remocn",
            doc_path=str(doc_path) if doc_path.exists() else None,
        )
        session.add(effect)

    session.commit()

count = session.exec(text("SELECT COUNT(*) FROM effect")).scalar_one()
print(f"Seeded {count} effects into database")
